---
# Spark Gap, Thermal COP, and Change in Demand Analysis
---

**Author:** Adapted from Jordan M. Joseph, PhD analysis framework  
**Affiliation:** Carnegie Mellon University  

## Overview

This notebook computes key adoption economics metrics for heat pump retrofits
by loading EUSS data directly with original column names.

**Workflow:**
1. **Load or Run:** Check for past exported results; if found, load them and skip computation
2. **EUSS Data Loading** (Step 1b): Load baseline and upgrade CSVs, filter to occupied SF homes
3. **Spark Gap** (Step 2): Electricity-to-natural gas price ratio by state ($/MMBTU)
4. **Thermal COP & Baseline AFUE** (Step 3): True HP efficiency and furnace efficiency from heating load data
5. **Bill Impact Ratio** (Step 4): Composite adoption KPI = spark_gap × (AFUE / COP)
6. **Change in Demand** (Step 5): Heating electricity demand change under 100% adoption
7. **Export Results:** Save results via the TARE model export system
8. **Geospatial Visualization** (Step 6): Choropleth maps of spark gap, bill impact ratio, COP, and demand change

**Batch Mode:** This notebook can be run interactively or called from the orchestration
notebook (`tare_run_simulation_v2_2.ipynb`) via `%run -i`. When the variable
`input_measure_package` is set before execution, interactive prompts are skipped.

**Data Sources:**
- NREL EUSS baseline and upgrade CSVs (ResStock AMY2018 Release 1.1)
- EIA fuel prices via `fuel_prices_nominal.csv`

**Key Interpretation:**
- `bill_impact_ratio < 1` → electrification saves money on heating bills
- `bill_impact_ratio > 1` → heat pump costs more to operate than gas furnace

**REVIEW THE README BEFORE USING:** Ensure the software environment and data folder are set up properly.

---
# STEP 0: Import Libraries and Configure Settings
---

**KEY LIBRARIES:**
- `pandas`: Tabular data processing
- `geopandas`: Geographic data (shapefiles and mapping)
- `matplotlib`: Visualization
- `numpy`: Numerical calculations

**IMPORTANT:** Always run this cell first!

In [ ]:
# Standard library imports
import os
from pathlib import Path
from typing import Dict, Optional, Tuple
from datetime import datetime

# Data processing
import pandas as pd
import numpy as np

# Geographic and visualization
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Project configuration
from config import PROJECT_ROOT

# TARE model constants (single source of truth)
from cmu_tare_model.constants import (
    ALLOWED_HOUSING_TYPES,
    VALID_MENU_MPS,
    VERBOSE,
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

print("✓ STEP 0 COMPLETE - All libraries loaded successfully!")
print(f"Project root: {PROJECT_ROOT}")
print(f"Allowed housing types: {ALLOWED_HOUSING_TYPES}")
print(f"Valid measure packages: {VALID_MENU_MPS}")

---
# STEP 0b: Batch Mode Detection and Measure Package Selection
---

**Batch Mode:** When called from `tare_run_simulation_v2_2.ipynb` via `%run -i`,
the variable `input_measure_package` is pre-set. Otherwise, the user selects interactively.

**Measure Packages:** Uses `VALID_MENU_MPS` from `constants.py` (excludes MP0 baseline).

In [ ]:
# ============================================================================
# BATCH MODE DETECTION AND MEASURE PACKAGE SELECTION
# ============================================================================

# Selectable MPs: all valid MPs except baseline (0)
SELECTABLE_MPS = [mp for mp in VALID_MENU_MPS if mp != 0]

def mp_to_upgrade(mp_num):
    """Convert MP number to EUSS upgrade string (e.g., 4 -> 'upgrade04', 10 -> 'upgrade10')."""
    return f"upgrade{mp_num:02d}"

# Check if batch mode (called from orchestration notebook)
try:
    _ = input_measure_package
    batch_mode = True
    selected_mps = [int(input_measure_package)]
    print(f"BATCH MODE: Running for MP{selected_mps[0]}")
except NameError:
    batch_mode = False
    # Interactive mode: select measure packages
    print(f"Available measure packages: {SELECTABLE_MPS}")
    mp_input = input(f"Enter MP numbers to analyze (comma-separated, or 'all'): ").strip()
    if mp_input.lower() == 'all':
        selected_mps = SELECTABLE_MPS
    else:
        selected_mps = [int(x.strip()) for x in mp_input.split(',') if x.strip().isdigit()]
        selected_mps = [mp for mp in selected_mps if mp in SELECTABLE_MPS]

    if not selected_mps:
        selected_mps = [4]  # Default to MP4
        print(f"No valid MPs selected. Defaulting to MP4.")

print(f"\nSelected measure packages: {selected_mps}")
print(f"Upgrade files: {[mp_to_upgrade(mp) for mp in selected_mps]}")

# ============================================================================
# CONVERSION CONSTANTS (Source: EIA)
# ============================================================================

BTU_PER_CF_NATURAL_GAS = 1039  # BTU per cubic foot of natural gas
BTU_PER_KWH = 3412              # BTU per kilowatt-hour (by definition)

# Derived conversion factor: natural gas $/1000cf to $/kWh
NG_CONVERSION_FACTOR = BTU_PER_KWH / (1000 * BTU_PER_CF_NATURAL_GAS)

# MMBTU-to-kWh conversion (1 MMBTU = 293.07107 kWh)
KWH_PER_MMBTU = 293.07107

# kBtu-to-kWh conversion (1 kWh = 3.412 kBtu, by definition from BTU_PER_KWH)
KBTU_PER_KWH = 3.412

# EUSS sampling weight: each surveyed home represents 242 dwelling units
DWELLING_UNIT_WEIGHT = 242

# Map EUSS heating energy columns to their corresponding price columns
FUEL_PRICE_MAP = {
    'out.electricity.heating.energy_consumption.kwh': 'elec_price_kwh',
    'out.natural_gas.heating.energy_consumption.kwh': 'gas_price_kwh',
}

print(f"\nNatural Gas Conversion Factor: {NG_CONVERSION_FACTOR:.6f}")
print(f"KWH_PER_MMBTU: {KWH_PER_MMBTU}")
print(f"KBTU_PER_KWH: {KBTU_PER_KWH}")

# ============================================================================
# STATE NAME MAPPING
# ============================================================================

STATE_NAMES = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'DC': 'District of Columbia', 'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii',
    'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine',
    'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska',
    'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico',
    'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island',
    'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas',
    'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington',
    'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

# Default adoption tiers for demand scenarios
DEFAULT_ADOPTER_TIERS = [
    'Tier 1: Feasible',
    'Tier 2: Feasible vs. Alternative'
]

print(f"\n✓ STEP 0b COMPLETE - Constants defined, MP selection: {selected_mps}")

---
# STEP 1b: Load EUSS Data
---

**Filters applied** (same as TARE model scenario notebooks):
- `in.vacancy_status == 'Occupied'` (exclude vacant homes)
- `in.geometry_building_type_recs ∈ ALLOWED_HOUSING_TYPES` (from constants.py)
- Upgrade files: `applicability == True` (measure was applicable to this building)

**Key EUSS columns used:**
| Column | Description |
|--------|-------------|
| `in.state` | 2-letter state code |
| `in.geometry_building_type_recs` | Housing type (RECS categories) |
| `in.vacancy_status` | Occupied or Vacant |
| `in.heating_fuel` | Primary heating fuel |
| `out.electricity.heating.energy_consumption.kwh` | Electricity consumed by heating (HP compressor) |
| `out.electricity.heating_hp_bkup.energy_consumption.kwh` | Electricity consumed by HP backup resistance |
| `out.electricity.heating_fans_pumps.energy_consumption.kwh` | Electricity consumed by supply fan/circulating pump during heating |
| `out.natural_gas.heating.energy_consumption.kwh` | Natural gas consumed by heating (kWh) |
| `out.fuel_oil.heating.energy_consumption.kwh` | Fuel oil consumed by heating (kWh) |
| `out.propane.heating.energy_consumption.kwh` | Propane consumed by heating (kWh) |
| `out.load.heating.energy_delivered.kbtu` | Thermal energy delivered to space (kBtu) |
| `weight` | Number of dwelling units this simulation represents |
| `applicability` | Whether the upgrade measure was applied (upgrade files only) |

**Note:** All energy consumption columns are in kWh. The heating load column is in kBtu.
Fan/pump electricity is included in the COP denominator because the heating load (numerator)
includes fan motor heat delivered to the space.

In [ ]:
# ============================================================================
# EUSS DATA PATHS AND COLUMN DEFINITIONS
# ============================================================================

# Path construction follows the same pattern as tare_scenarios_v2_2.ipynb
EUSS_DATA_DIR = os.path.join(
    PROJECT_ROOT, "cmu_tare_model", "data", "euss_data",
    "resstock_amy2018_release_1.1", "national", "csv"
)

# Original EUSS heating energy columns (all in kWh)
HEATING_FUEL_COLS = [
    'out.electricity.heating.energy_consumption.kwh',
    'out.natural_gas.heating.energy_consumption.kwh',
    'out.fuel_oil.heating.energy_consumption.kwh',
    'out.propane.heating.energy_consumption.kwh',
]

# Heating load (thermal energy delivered to space) and auxiliary electricity columns
HEATING_LOAD_COL = 'out.load.heating.energy_delivered.kbtu'
HP_BACKUP_ELEC_COL = 'out.electricity.heating_hp_bkup.energy_consumption.kwh'
HP_FANS_PUMPS_COL = 'out.electricity.heating_fans_pumps.energy_consumption.kwh'

# Columns to load from EUSS CSVs (subset for efficiency)
BASELINE_USECOLS = [
    'bldg_id', 'in.state', 'in.vacancy_status',
    'in.geometry_building_type_recs',
    'in.heating_fuel', 'in.hvac_heating_type_and_fuel',
    'in.hvac_heating_efficiency', 'weight',
] + HEATING_FUEL_COLS + [HEATING_LOAD_COL, HP_BACKUP_ELEC_COL, HP_FANS_PUMPS_COL]

UPGRADE_USECOLS = BASELINE_USECOLS + ['applicability']


# ============================================================================
# LOAD EUSS BASELINE AND UPGRADE DATA
# ============================================================================

def load_euss_baseline(filename='baseline_metadata_and_annual_results.csv'):
    """Load EUSS baseline CSV, filter to occupied single-family homes."""
    filepath = os.path.join(EUSS_DATA_DIR, filename)
    print(f"Loading baseline from: {filepath}")
    df = pd.read_csv(filepath, usecols=BASELINE_USECOLS, index_col='bldg_id')

    # Apply same 3-stage filter pipeline as scenario notebooks
    n_total = len(df)
    df = df[df['in.vacancy_status'] == 'Occupied']
    print(f"  After occupancy filter: {len(df):,} / {n_total:,}")

    df = df[df['in.geometry_building_type_recs'].isin(ALLOWED_HOUSING_TYPES)]
    print(f"  After housing type filter ({ALLOWED_HOUSING_TYPES}): {len(df):,}")

    return df


def load_euss_upgrade(upgrade_name):
    """Load EUSS upgrade CSV, filter to occupied SF homes where measure was applicable."""
    filename = f'{upgrade_name}_metadata_and_annual_results.csv'
    filepath = os.path.join(EUSS_DATA_DIR, filename)
    print(f"Loading {upgrade_name} from: {filepath}")
    df = pd.read_csv(filepath, usecols=UPGRADE_USECOLS, index_col='bldg_id')

    # Apply same 3-stage filter pipeline as scenario notebooks
    n_total = len(df)
    df = df[df['in.vacancy_status'] == 'Occupied']
    print(f"  After occupancy filter: {len(df):,} / {n_total:,}")

    df = df[df['in.geometry_building_type_recs'].isin(ALLOWED_HOUSING_TYPES)]
    print(f"  After housing type filter: {len(df):,}")

    df = df[df['applicability'] == True]
    print(f"  After applicability filter: {len(df):,}")

    return df


# ============================================================================
# LOAD DATA FOR ALL SELECTED MEASURE PACKAGES
# ============================================================================

print("=" * 80)
print("STEP 1b: LOAD EUSS DATA")
print("=" * 80)

# Load baseline (shared across all MPs)
print("\nLoading EUSS baseline data...")
df_baseline = load_euss_baseline()
print(f"  Baseline: {len(df_baseline):,} occupied SF homes")

# Load upgrades for each selected MP
upgrade_data = {}  # Dict[int, pd.DataFrame] — keyed by MP number
for mp in selected_mps:
    upgrade_name = mp_to_upgrade(mp)
    print(f"\nLoading EUSS upgrade data for MP{mp} ({upgrade_name})...")
    upgrade_data[mp] = load_euss_upgrade(upgrade_name)
    print(f"  MP{mp}: {len(upgrade_data[mp]):,} applicable homes")

# Quick diagnostic: verify fan/pump column exists
print(f"\n--- Fan/Pump Column Check ---")
print(f"  Baseline '{HP_FANS_PUMPS_COL}' non-null: {df_baseline[HP_FANS_PUMPS_COL].notna().sum():,}")
for mp, df_up in upgrade_data.items():
    print(f"  MP{mp} '{HP_FANS_PUMPS_COL}' non-null: {df_up[HP_FANS_PUMPS_COL].notna().sum():,}")

print(f"\n✓ STEP 1b COMPLETE - EUSS data loaded for MPs: {selected_mps}")

---
# STEP 2: Load and Process Fuel Price Data (CSV Approach)
---

**PROCESS:**
1. Load raw EIA price data from CSV (nominal prices)
2. Filter to state-level electricity and natural gas prices
3. Convert both prices to $/kWh for fair comparison
4. Convert both prices to $/MMBTU (industry-standard unit for spark gap)
5. Calculate the spark gap (electricity-to-gas price ratio)

**OUTPUT:**
A DataFrame with columns:
- `state`: Two-letter state code
- `state_name`: Full state name
- `elec_price_kwh`: Electricity price ($/kWh, nominal)
- `gas_price_kwh`: Natural gas price ($/kWh equivalent, nominal)
- `elec_price_mmbtu`: Electricity price ($/MMBTU, nominal)
- `gas_price_mmbtu`: Natural gas price ($/MMBTU, nominal)
- `spark_gap`: Price ratio = elec_price_mmbtu / gas_price_mmbtu

**Note:** This uses *nominal* prices from the raw CSV. For inflation-adjusted USD2023
prices with AEO projections, see Step 2b which uses the TARE model's lookup dict.

In [ ]:
def calculate_price_ratios(
    filepath: str,
    year: int = 2022
) -> pd.DataFrame:
    """
    Load fuel price data and calculate electricity-to-gas price ratios by state.

    Reads nominal EIA prices from CSV, converts to $/kWh and $/MMBTU,
    and computes the spark gap (electricity/gas price ratio).

    Args:
        filepath: Path to fuel_prices_nominal.csv file.
        year: Year for price comparison (default: 2022).

    Returns:
        DataFrame with state-level price comparisons, $/MMBTU values, and spark gap.
    """
    df = pd.read_csv(filepath)
    price_col = f'{year}_nominal_unit_price'

    if price_col not in df.columns:
        raise KeyError(
            f"Column '{price_col}' not found. "
            f"Available year columns: {[c for c in df.columns if 'nominal_unit_price' in c]}"
        )

    # Extract natural gas prices (state-level only, skip National and census divisions)
    df_ng = df[
        (df['fuel_type'] == 'naturalGas') &
        (df['state_region'].str.len() == 2) &
        (df['state_region'] != 'National')
    ][['state_region', price_col]].copy()
    df_ng.columns = ['state', 'ng_price_per_1000cf']

    # Extract electricity prices (state-level only)
    df_elec = df[
        (df['fuel_type'] == 'electricity') &
        (df['state_region'].str.len() == 2) &
        (df['state_region'] != 'National')
    ][['state_region', price_col]].copy()
    df_elec.columns = ['state', 'elec_price_cents_kwh']

    # Merge datasets
    df_merged = df_elec.merge(df_ng, on='state', how='inner')

    # Convert to $/kWh
    df_merged['elec_price_kwh'] = df_merged['elec_price_cents_kwh'] / 100
    df_merged['gas_price_kwh'] = df_merged['ng_price_per_1000cf'] * NG_CONVERSION_FACTOR

    # Convert to $/MMBTU (industry standard for spark gap comparisons)
    df_merged['elec_price_mmbtu'] = df_merged['elec_price_kwh'] * KWH_PER_MMBTU
    df_merged['gas_price_mmbtu'] = df_merged['gas_price_kwh'] * KWH_PER_MMBTU

    # Spark gap = electricity/gas price ratio on energy-equivalent basis
    df_merged['spark_gap'] = df_merged['elec_price_mmbtu'] / df_merged['gas_price_mmbtu']

    # Add state names
    df_merged['state_name'] = df_merged['state'].map(STATE_NAMES)

    # Select and order columns
    result = df_merged[[
        'state', 'state_name',
        'elec_price_kwh', 'gas_price_kwh',
        'elec_price_mmbtu', 'gas_price_mmbtu',
        'spark_gap'
    ]].copy()

    # Round for display
    result['elec_price_kwh'] = result['elec_price_kwh'].round(4)
    result['gas_price_kwh'] = result['gas_price_kwh'].round(4)
    result['elec_price_mmbtu'] = result['elec_price_mmbtu'].round(2)
    result['gas_price_mmbtu'] = result['gas_price_mmbtu'].round(2)
    result['spark_gap'] = result['spark_gap'].round(2)

    return result.sort_values('spark_gap', ascending=False).reset_index(drop=True)


# ============================================================================
# LOAD AND PROCESS DATA
# ============================================================================

FUEL_PRICES_PATH = os.path.join(
    PROJECT_ROOT, "cmu_tare_model", "data", "fuel_prices", "fuel_prices_nominal.csv"
)

print("===== STEP 2: LOAD AND PROCESS FUEL PRICE DATA (CSV) =====")
df_prices_csv = calculate_price_ratios(FUEL_PRICES_PATH, year=2022)
print(f"✓ Loaded price data for {len(df_prices_csv)} states")

# Display summary statistics
print(f"\n--- Summary Statistics ---")
print(f"Mean Spark Gap:   {df_prices_csv['spark_gap'].mean():.2f}")
print(f"Median Spark Gap: {df_prices_csv['spark_gap'].median():.2f}")
print(f"Range:            {df_prices_csv['spark_gap'].min():.2f} - {df_prices_csv['spark_gap'].max():.2f}")

print(f"\n--- Top 5 States (Highest Spark Gap - Electricity Relatively Most Expensive) ---")
print(df_prices_csv.head(5).to_string(index=False))

print(f"\n--- Bottom 5 States (Lowest Spark Gap - Electricity Relatively Least Expensive) ---")
print(df_prices_csv.tail(5).to_string(index=False))

# ============================================================================
# PLACEHOLDER: 10-Year Average Residential Fuel Prices
# ============================================================================
# TODO: Load updated 10-year average residential fuel price data when available.
# This will provide a more stable price baseline by averaging across price
# cycles rather than relying on a single year's snapshot.
#
# df_prices_10yr_avg = calculate_price_ratios_10yr_avg(filepath, start_year, end_year)
#
print("\n⚠ 10-year average prices: PLACEHOLDER — awaiting updated data")

---
# STEP 2b: Load Fuel Prices from TARE Model Lookup Dict (Alternative)
---

**Why two approaches?**
- **Step 2 (CSV):** Reads raw nominal EIA prices directly from CSV. Simple and self-contained.
- **Step 2b (Lookup):** Uses the TARE model's pre-processed `lookup_fuel_prices_iraRef` dict, which is already inflation-adjusted to USD2023 and includes AEO projections out to 2050.

The lookup dict structure is:
```
lookup[state_abbr][fuel_type][policy_scenario][year] → price in USD2023/kWh
```

Both approaches produce a DataFrame with the same columns (`state`, `elec_price_kwh`, `gas_price_kwh`, `elec_price_mmbtu`, `gas_price_mmbtu`, `spark_gap`), but on different price bases.

In [ ]:
def get_state_fuel_prices_from_lookup(
    lookup: Dict,
    year: int = 2022,
    policy_scenario: str = 'AEO2023 Reference Case'
) -> pd.DataFrame:
    """
    Extract state-level fuel prices from the TARE model's pre-processed lookup dict.

    The lookup dict contains inflation-adjusted USD2023/kWh prices with AEO projections.
    This function filters to 2-character state keys (skipping census divisions and 'National'),
    extracts electricity and natural gas prices, and converts to $/MMBTU.

    Args:
        lookup: Nested dict from create_lookup_fuel_prices.py
            (e.g., lookup_fuel_prices_iraRef). Structure:
            lookup[state][fuel_type][policy_scenario][year] → USD2023/kWh.
        year: Reference year for price extraction (default: 2022).
        policy_scenario: AEO scenario string (default: 'AEO2023 Reference Case').

    Returns:
        DataFrame with columns: state, elec_price_kwh, gas_price_kwh,
        elec_price_mmbtu, gas_price_mmbtu, spark_gap.

    Raises:
        KeyError: If no valid state entries are found in the lookup dict.
    """
    rows = []
    for location in lookup:
        # State entries are exactly 2 characters (e.g., 'PA', 'FL')
        # Skip census divisions (longer strings) and 'National'
        if len(location) != 2:
            continue

        elec_price = (
            lookup[location]
            .get('electricity', {})
            .get(policy_scenario, {})
            .get(year, np.nan)
        )
        gas_price = (
            lookup[location]
            .get('naturalGas', {})
            .get(policy_scenario, {})
            .get(year, np.nan)
        )

        rows.append({
            'state': location,
            'elec_price_kwh': elec_price,
            'gas_price_kwh': gas_price,
        })

    if not rows:
        raise KeyError(
            f"No 2-character state entries found in lookup dict. "
            f"Available keys: {list(lookup.keys())[:10]}..."
        )

    df = pd.DataFrame(rows)

    # Convert to $/MMBTU
    df['elec_price_mmbtu'] = df['elec_price_kwh'] * KWH_PER_MMBTU
    df['gas_price_mmbtu'] = df['gas_price_kwh'] * KWH_PER_MMBTU

    # Spark gap = electricity/gas price ratio
    df['spark_gap'] = df['elec_price_mmbtu'] / df['gas_price_mmbtu']

    # Round for display
    df['elec_price_kwh'] = df['elec_price_kwh'].round(6)
    df['gas_price_kwh'] = df['gas_price_kwh'].round(6)
    df['elec_price_mmbtu'] = df['elec_price_mmbtu'].round(2)
    df['gas_price_mmbtu'] = df['gas_price_mmbtu'].round(2)
    df['spark_gap'] = df['spark_gap'].round(2)

    return df.sort_values('spark_gap', ascending=False).reset_index(drop=True)


# ============================================================================
# LOAD PRICES FROM TARE MODEL LOOKUP DICT
# ============================================================================

# Import the pre-processed lookup dict (USD2023/kWh with AEO projections)
from cmu_tare_model.private_impact.data_processing.create_lookup_fuel_prices import (
    lookup_fuel_prices_iraRef
)

print("===== STEP 2b: LOAD FUEL PRICES FROM LOOKUP DICT (USD2023) =====")
df_prices_lookup = get_state_fuel_prices_from_lookup(
    lookup_fuel_prices_iraRef, year=2022, policy_scenario='AEO2023 Reference Case'
)
print(f"✓ Extracted prices for {len(df_prices_lookup)} states from lookup dict")

# Check for missing data
n_missing = df_prices_lookup[['elec_price_kwh', 'gas_price_kwh']].isna().any(axis=1).sum()
if n_missing > 0:
    print(f"⚠ {n_missing} states have missing price data")
else:
    print(f"✓ No missing price data — all {len(df_prices_lookup)} states have complete records")

print(f"\n--- Summary Statistics (USD2023 prices) ---")
print(f"Mean Spark Gap:   {df_prices_lookup['spark_gap'].mean():.2f}")
print(f"Median Spark Gap: {df_prices_lookup['spark_gap'].median():.2f}")
print(f"Range:            {df_prices_lookup['spark_gap'].min():.2f} - {df_prices_lookup['spark_gap'].max():.2f}")

print(f"\n--- Top 5 States (Highest Spark Gap) ---")
print(df_prices_lookup.head(5).to_string(index=False))

print(f"\n--- Bottom 5 States (Lowest Spark Gap) ---")
print(df_prices_lookup.tail(5).to_string(index=False))

---
# STEP 3: Compute Thermal COP and Baseline AFUE by State
---

**Method (from literature):** Compute building-stock-weighted heat pump COP by dividing
the aggregate space heat delivered by the aggregate electric heating input, at the state level.

**Thermal COP** = Σ(Q_upgrade) / Σ(E_hp + E_backup + E_fans_pumps) per state
- `Q_upgrade`: `out.load.heating.energy_delivered.kbtu` (heat delivered to space by HP)
- `E_hp`: `out.electricity.heating.energy_consumption.kwh` (HP compressor)
- `E_backup`: `out.electricity.heating_hp_bkup.energy_consumption.kwh` (backup resistance)
- `E_fans_pumps`: `out.electricity.heating_fans_pumps.energy_consumption.kwh` (supply fan / circulating pump)

**Why include fan/pump electricity?** The heating load column (`out.load.heating.energy_delivered.kbtu`)
includes fan motor heat delivered to the space. Excluding fan/pump electricity from the denominator
while including its thermal contribution in the numerator inflates COP — especially in warm states
where fan energy is a larger fraction of total heating electricity.

**Baseline AFUE** = Σ(Q_baseline) / Σ(F_gas) per state
- `Q_baseline`: `out.load.heating.energy_delivered.kbtu` (heat delivered by furnace)
- `F_gas`: `out.natural_gas.heating.energy_consumption.kwh` (gas consumed)
- Serves as validation against metadata AFUE values

Both quantities computed from the same underlying heating load column,
ensuring apples-to-apples comparison.

**Expected ranges:**
- Thermal COP: **2.0–4.5** (cold states ~2.0–3.0, warm states ~3.5–4.5)
- Baseline AFUE: **0.60–0.85** (should correlate with `in.hvac_heating_efficiency` metadata)

**Fuel filtering (default: Natural Gas only):**
The spark gap metric is specifically about gas-to-electric economics.
Only gas-heated homes are included by default.

In [ ]:
def compute_thermal_cop_by_state(
    df_baseline: pd.DataFrame,
    df_upgrade: pd.DataFrame,
    fuel_filter: str = 'Natural Gas',
    verbose: bool = False
) -> pd.DataFrame:
    """
    Compute state-level thermal COP and baseline furnace AFUE from EUSS data.

    Uses the heating load column (out.load.heating.energy_delivered.kbtu) to
    compute the true thermal COP following the literature method: aggregate
    space heat delivered divided by aggregate electric heating input per state.

    The denominator includes HP compressor, backup resistance, AND supply
    fan/pump electricity — because the numerator (heating load) includes
    fan motor heat delivered to the space.

    Also computes baseline furnace AFUE from the same heating load column
    divided by gas consumed, providing a data-driven efficiency rather than
    an assumed constant.

    Args:
        df_baseline: EUSS baseline DataFrame (indexed by bldg_id) with columns
            including HEATING_LOAD_COL, gas consumption, 'in.state', 'in.heating_fuel'.
        df_upgrade: EUSS upgrade DataFrame (indexed by bldg_id, already filtered
            to applicability == True) with HEATING_LOAD_COL, electricity,
            backup electricity, and fan/pump electricity columns.
        fuel_filter: Value of 'in.heating_fuel' to filter to (default: 'Natural Gas').
            Set to None to include all fuel types.
        verbose: Print diagnostic info.

    Returns:
        DataFrame with columns: state, thermal_cop, baseline_afue,
        Q_upgrade_total_kbtu, hp_total_elec_kbtu, Q_baseline_total_kbtu,
        gas_consumed_total_kbtu, fans_pumps_pct, home_count.

    Raises:
        KeyError: If required columns are missing from input DataFrames.
    """
    elec_col = 'out.electricity.heating.energy_consumption.kwh'
    bkup_col = HP_BACKUP_ELEC_COL
    fans_col = HP_FANS_PUMPS_COL
    load_col = HEATING_LOAD_COL
    gas_col = 'out.natural_gas.heating.energy_consumption.kwh'

    # Validate required columns
    for col, name in [(load_col, 'baseline'), (gas_col, 'baseline')]:
        if col not in df_baseline.columns:
            raise KeyError(f"Missing column '{col}' in {name} DataFrame")
    for col, name in [(load_col, 'upgrade'), (elec_col, 'upgrade'),
                       (bkup_col, 'upgrade'), (fans_col, 'upgrade')]:
        if col not in df_upgrade.columns:
            raise KeyError(f"Missing column '{col}' in {name} DataFrame")

    # Build merged DataFrame with baseline and upgrade data (inner join on bldg_id)
    df_merged = pd.DataFrame({
        'state': df_baseline['in.state'],
        'heating_fuel': df_baseline['in.heating_fuel'],
        'Q_baseline_kbtu': df_baseline[load_col].fillna(0),
        'gas_consumed_kwh': df_baseline[gas_col].fillna(0),
    }).join(
        pd.DataFrame({
            'Q_upgrade_kbtu': df_upgrade[load_col].fillna(0),
            'hp_elec_kwh': df_upgrade[elec_col].fillna(0),
            'hp_bkup_elec_kwh': df_upgrade[bkup_col].fillna(0),
            'hp_fans_pumps_kwh': df_upgrade[fans_col].fillna(0),
        }),
        how='inner'
    )

    if verbose:
        print(f"Matched homes (baseline ∩ upgrade): {len(df_merged):,}")

    # Apply fuel filter
    if fuel_filter is not None:
        n_before = len(df_merged)
        df_merged = df_merged[df_merged['heating_fuel'] == fuel_filter]
        if verbose:
            print(f"Filtered to '{fuel_filter}': {len(df_merged):,} / {n_before:,} homes "
                  f"({100*len(df_merged)/n_before:.1f}%)")

    # Total retrofit heating electricity: HP compressor + backup resistance + fan/pump
    df_merged['hp_total_elec_kwh'] = (
        df_merged['hp_elec_kwh']
        + df_merged['hp_bkup_elec_kwh']
        + df_merged['hp_fans_pumps_kwh']
    )

    # Convert electricity and gas to kBtu for consistent units with heating load
    df_merged['hp_total_elec_kbtu'] = df_merged['hp_total_elec_kwh'] * KBTU_PER_KWH
    df_merged['hp_fans_pumps_kbtu'] = df_merged['hp_fans_pumps_kwh'] * KBTU_PER_KWH
    df_merged['gas_consumed_kbtu'] = df_merged['gas_consumed_kwh'] * KBTU_PER_KWH

    # Aggregate by state
    grouped = df_merged.groupby('state').agg(
        Q_upgrade_total_kbtu=('Q_upgrade_kbtu', 'sum'),
        hp_total_elec_kbtu=('hp_total_elec_kbtu', 'sum'),
        hp_fans_pumps_total_kbtu=('hp_fans_pumps_kbtu', 'sum'),
        Q_baseline_total_kbtu=('Q_baseline_kbtu', 'sum'),
        gas_consumed_total_kbtu=('gas_consumed_kbtu', 'sum'),
        home_count=('state', 'size')
    ).reset_index()

    # Thermal COP: heat delivered by HP system / total electricity consumed
    # Denominator includes compressor + backup + fan/pump electricity
    grouped['thermal_cop'] = np.where(
        grouped['hp_total_elec_kbtu'] > 0,
        grouped['Q_upgrade_total_kbtu'] / grouped['hp_total_elec_kbtu'],
        np.nan
    )

    # Baseline furnace efficiency: heat delivered / gas consumed (both in kBtu)
    grouped['baseline_afue'] = np.where(
        grouped['gas_consumed_total_kbtu'] > 0,
        grouped['Q_baseline_total_kbtu'] / grouped['gas_consumed_total_kbtu'],
        np.nan
    )

    # Fan/pump energy as % of total HP electricity (diagnostic)
    grouped['fans_pumps_pct'] = np.where(
        grouped['hp_total_elec_kbtu'] > 0,
        grouped['hp_fans_pumps_total_kbtu'] / grouped['hp_total_elec_kbtu'] * 100,
        0
    )

    if verbose:
        fuel_label = fuel_filter if fuel_filter else 'all fuels'
        print(f"\n--- Thermal COP Summary ({fuel_label}) ---")
        print(f"States: {len(grouped)}")
        print(f"Mean:   {grouped['thermal_cop'].mean():.2f}")
        print(f"Median: {grouped['thermal_cop'].median():.2f}")
        print(f"Range:  {grouped['thermal_cop'].min():.2f} - {grouped['thermal_cop'].max():.2f}")

        # Flag suspicious values
        suspect_cop = grouped[
            (grouped['thermal_cop'] < 1.5) | (grouped['thermal_cop'] > 5.0)
        ]
        if len(suspect_cop) > 0:
            print(f"⚠ {len(suspect_cop)} states with suspicious COP (<1.5 or >5.0):")
            for _, row in suspect_cop.iterrows():
                print(f"    {row['state']}: {row['thermal_cop']:.2f}")
        else:
            print("✓ All states within expected COP range (1.5–5.0)")

        print(f"\n--- Baseline AFUE Summary ({fuel_label}) ---")
        print(f"Mean:   {grouped['baseline_afue'].mean():.2f}")
        print(f"Median: {grouped['baseline_afue'].median():.2f}")
        print(f"Range:  {grouped['baseline_afue'].min():.2f} - {grouped['baseline_afue'].max():.2f}")

        suspect_afue = grouped[
            (grouped['baseline_afue'] < 0.50) | (grouped['baseline_afue'] > 1.0)
        ]
        if len(suspect_afue) > 0:
            print(f"⚠ {len(suspect_afue)} states with suspicious AFUE (<0.50 or >1.0):")
            for _, row in suspect_afue.iterrows():
                print(f"    {row['state']}: {row['baseline_afue']:.2f}")

        print(f"\n--- Fan/Pump Energy as % of Total HP Electricity ---")
        print(f"Mean:   {grouped['fans_pumps_pct'].mean():.1f}%")
        print(f"Range:  {grouped['fans_pumps_pct'].min():.1f}% - {grouped['fans_pumps_pct'].max():.1f}%")

    return grouped

In [ ]:
# ============================================================================
# COMPUTE THERMAL COP AND BASELINE AFUE FOR ALL SELECTED MPs
# ============================================================================

# Store COP results per MP
cop_results = {}  # Dict[int, pd.DataFrame]

for mp in selected_mps:
    print(f"\n{'=' * 80}")
    print(f"STEP 3: COMPUTE THERMAL COP & BASELINE AFUE (MP{mp}, Natural Gas homes)")
    print(f"{'=' * 80}")

    cop_results[mp] = compute_thermal_cop_by_state(
        df_baseline, upgrade_data[mp], fuel_filter='Natural Gas', verbose=True
    )

    df_cop_mp = cop_results[mp]

    print(f"\n--- Top 5 States by Thermal COP (MP{mp}) ---")
    print(df_cop_mp.sort_values('thermal_cop', ascending=False).head(5)[
        ['state', 'thermal_cop', 'baseline_afue', 'fans_pumps_pct', 'home_count']
    ].to_string(index=False))

    print(f"\n--- Bottom 5 States by Thermal COP (MP{mp}) ---")
    print(df_cop_mp.sort_values('thermal_cop', ascending=True).head(5)[
        ['state', 'thermal_cop', 'baseline_afue', 'fans_pumps_pct', 'home_count']
    ].to_string(index=False))

    # Validation: heating load consistency between baseline and upgrade
    common_ids = df_baseline.index.intersection(upgrade_data[mp].index)
    Q_baseline = df_baseline.loc[common_ids, HEATING_LOAD_COL].fillna(0)
    Q_upgrade = upgrade_data[mp].loc[common_ids, HEATING_LOAD_COL].fillna(0)
    mask = Q_baseline > 0
    pct_diff = ((Q_upgrade[mask] - Q_baseline[mask]) / Q_baseline[mask]).median()
    print(f"\n--- Validation: Heating Load Consistency (MP{mp}) ---")
    print(f"Median Q difference (upgrade vs baseline): {pct_diff:.1%}")
    print(f"(Should be within +/-5-10% — small differences from distribution losses, unmet hours)")

# Use first selected MP as primary for subsequent steps
primary_mp = selected_mps[0]
df_cop = cop_results[primary_mp]
df_upgrade_primary = upgrade_data[primary_mp]

print(f"\n✓ STEP 3 COMPLETE — Primary MP for Steps 4-5: MP{primary_mp}")

---
# STEP 4: Composite Spark Gap Metrics (Bill Impact Ratio)
---

**Merges fuel prices (Step 2/2b) with thermal COP and baseline AFUE (Step 3) to compute:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| `spark_gap` | elec_price / gas_price ($/MMBTU) | Raw price ratio |
| `thermal_cop` | Q_delivered / E_hp_total | True heat pump thermal efficiency (2.0–4.5) |
| `baseline_afue` | Q_delivered / F_gas | Baseline furnace efficiency from data (0.60–0.85) |
| `bill_impact_ratio` | spark_gap × (AFUE / COP) | HP heat cost as fraction of furnace heat cost. < 1 = savings |

**Example:** If spark_gap = 3.5, thermal_cop = 3.2, baseline_afue = 0.76:
- `bill_impact_ratio` = 3.5 × (0.76 / 3.2) = 0.831 → heat pump heat costs 83.1% of furnace heat → **16.9% savings**

The **bill_impact_ratio** is the key adoption KPI:
- `< 1` → heat pump operating costs are lower than the gas furnace alternative
- `= 1` → cost-neutral
- `> 1` → heat pump operating costs are higher (adoption is subsidy-dependent)

**Note:** The AFUE now comes directly from the EUSS simulation data (not an assumed constant),
ensuring consistent treatment of both the heat pump and furnace efficiencies.

In [ ]:
def compute_spark_gap_metrics(
    df_prices: pd.DataFrame,
    df_cop: pd.DataFrame,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Merge fuel prices with thermal COP and baseline AFUE to compute bill impact ratio.

    Combines the spark gap (price ratio) with building-stock-weighted thermal COP
    and data-derived baseline AFUE from compute_thermal_cop_by_state() to produce
    the bill_impact_ratio — the heat pump heating cost as a fraction of the
    incumbent gas furnace heating cost.

    Args:
        df_prices: State-level fuel prices with columns: state, spark_gap,
            elec_price_kwh, gas_price_kwh, elec_price_mmbtu, gas_price_mmbtu.
        df_cop: State-level COP/AFUE with columns: state, thermal_cop,
            baseline_afue, home_count. Should be from gas-heated subset.
        verbose: Print diagnostic info.

    Returns:
        Merged DataFrame with all input columns plus:
        - bill_impact_ratio: HP heating cost as fraction of gas furnace heating cost.
            < 1 → electrification saves money (e.g., 0.80 = 20% savings)
            = 1 → cost-neutral
            > 1 → electrification costs more (e.g., 1.15 = 15% increase)
    """
    # Validate inputs
    price_required = ['state', 'spark_gap']
    cop_required = ['state', 'thermal_cop', 'baseline_afue']
    missing_price = [c for c in price_required if c not in df_prices.columns]
    missing_cop = [c for c in cop_required if c not in df_cop.columns]
    if missing_price:
        raise KeyError(f"Missing columns in df_prices: {missing_price}")
    if missing_cop:
        raise KeyError(f"Missing columns in df_cop: {missing_cop}")

    # Merge on state
    df = df_prices.merge(
        df_cop[['state', 'thermal_cop', 'baseline_afue', 'home_count',
                'Q_upgrade_total_kbtu', 'hp_total_elec_kbtu']],
        on='state',
        how='inner'
    )

    if verbose:
        n_price = len(df_prices['state'].unique())
        n_cop = len(df_cop['state'].unique())
        n_merged = len(df['state'].unique())
        print(f"Merge: {n_price} price states × {n_cop} COP states → {n_merged} matched")

    # Bill impact ratio: cost of HP heat / cost of furnace heat
    # = (elec_price / thermal_cop) / (gas_price / baseline_afue)
    # = spark_gap × (baseline_afue / thermal_cop)
    # AFUE appears exactly once, derived from data — no double-counting
    df['bill_impact_ratio'] = df['spark_gap'] * (df['baseline_afue'] / df['thermal_cop'])

    if verbose:
        print(f"\nThermal COP: {df['thermal_cop'].mean():.2f} (mean)")
        print(f"Baseline AFUE: {df['baseline_afue'].mean():.2f} (mean)")
        n_favorable = (df['bill_impact_ratio'] < 1).sum()
        print(f"\nBill Impact Ratio (HP cost / furnace cost):")
        print(f"  States where electrification saves money: {n_favorable} / {len(df)}")
        print(f"  Mean: {df['bill_impact_ratio'].mean():.2f}")
        print(f"  Range: {df['bill_impact_ratio'].min():.2f} - {df['bill_impact_ratio'].max():.2f}")

    return df.sort_values('bill_impact_ratio', ascending=True).reset_index(drop=True)


# ============================================================================
# COMPUTE SPARK GAP METRICS (using thermal COP and data-derived AFUE)
# ============================================================================

print(f"===== STEP 4: COMPOSITE SPARK GAP METRICS (MP{primary_mp}) =====")
df_spark = compute_spark_gap_metrics(df_prices_csv, df_cop, verbose=True)

print(f"\n--- Top 5 States (Best for Electrification) ---")
print(df_spark[['state', 'spark_gap', 'thermal_cop', 'baseline_afue', 'bill_impact_ratio']].head(5).to_string(index=False))

print(f"\n--- Bottom 5 States (Worst for Electrification) ---")
print(df_spark[['state', 'spark_gap', 'thermal_cop', 'baseline_afue', 'bill_impact_ratio']].tail(5).to_string(index=False))

print("\n✓ STEP 4 COMPLETE")

---
# STEP 4b: Actual Bill Savings by State (Fuel-Specific Pricing)
---

**Validation metric:** Computes per-building heating bill change using actual fuel-specific prices,
rather than the ratio-based shortcut (`bill_impact_ratio = spark_gap × AFUE / COP`).

**Why this matters:** The ratio-based metric uses state-average COP and AFUE. This direct
calculation computes per-building costs and then aggregates, serving as ground truth.

```
baseline_cost = Σ(electricity_kwh × elec_price + gas_kwh × gas_price)
retrofit_cost = Σ(retrofit_elec_total_kwh × elec_price)
bill_savings  = baseline_cost - retrofit_cost
```

**Note:** Retrofit electricity includes both HP compressor and backup resistance.
Only electricity and natural gas prices are available. Propane and fuel oil homes are
excluded unless those price data are added later.

In [ ]:
def compute_actual_bill_savings_by_state(
    df_baseline: pd.DataFrame,
    df_upgrade: pd.DataFrame,
    df_prices: pd.DataFrame,
    fuel_filter: str = 'Natural Gas',
    verbose: bool = False
) -> pd.DataFrame:
    """
    Compute state-level bill savings using actual fuel-specific prices per building.

    Unlike the ratio-based bill_impact_ratio, this directly computes baseline
    and retrofit heating costs per building using the correct price for each fuel,
    then aggregates by state. Serves as ground truth for validation.

    Retrofit electricity includes HP compressor, backup resistance, and supply
    fan/pump electricity — all are metered and billed to the homeowner.

    Args:
        df_baseline: EUSS baseline DataFrame (indexed by bldg_id) with
            HEATING_FUEL_COLS, 'in.state', 'in.heating_fuel', and 'weight'.
        df_upgrade: EUSS upgrade DataFrame (indexed by bldg_id, filtered to
            applicability == True) with electricity, backup, and fan/pump columns.
        df_prices: State-level prices with columns: state, elec_price_kwh, gas_price_kwh.
        fuel_filter: Filter to this heating fuel (default: 'Natural Gas').
            Set to None to include all fuel types.
        verbose: Print diagnostic info.

    Returns:
        DataFrame with columns: state, home_count, total_baseline_cost,
        total_retrofit_cost, total_bill_savings, avg_savings_per_home,
        pct_bill_savings.
    """
    elec_col = 'out.electricity.heating.energy_consumption.kwh'
    bkup_col = HP_BACKUP_ELEC_COL
    fans_col = HP_FANS_PUMPS_COL
    gas_col = 'out.natural_gas.heating.energy_consumption.kwh'

    # Build per-building table with both fuels' consumption
    df_bills = pd.DataFrame({
        'in.state': df_baseline['in.state'],
        'in.heating_fuel': df_baseline['in.heating_fuel'],
        'weight': df_baseline['weight'],
        'baseline_elec_kwh': df_baseline[elec_col],
        'baseline_gas_kwh': df_baseline[gas_col],
    }).join(
        pd.DataFrame({
            'retrofit_hp_kwh': df_upgrade[elec_col].fillna(0),
            'retrofit_bkup_kwh': df_upgrade[bkup_col].fillna(0),
            'retrofit_fans_kwh': df_upgrade[fans_col].fillna(0),
        }),
        how='inner'
    )

    # Total retrofit electricity = HP compressor + backup resistance + fan/pump
    df_bills['retrofit_elec_kwh'] = (
        df_bills['retrofit_hp_kwh']
        + df_bills['retrofit_bkup_kwh']
        + df_bills['retrofit_fans_kwh']
    )

    if fuel_filter is not None:
        df_bills = df_bills[df_bills['in.heating_fuel'] == fuel_filter]

    # Join state-level prices
    df_bills = df_bills.merge(
        df_prices[['state', 'elec_price_kwh', 'gas_price_kwh']],
        left_on='in.state', right_on='state', how='inner'
    ).drop(columns='state')

    # Per-building costs ($/year)
    df_bills['baseline_cost'] = (
        df_bills['baseline_elec_kwh'] * df_bills['elec_price_kwh'] +
        df_bills['baseline_gas_kwh'] * df_bills['gas_price_kwh']
    )
    df_bills['retrofit_cost'] = (
        df_bills['retrofit_elec_kwh'] * df_bills['elec_price_kwh']
    )
    df_bills['bill_savings'] = df_bills['baseline_cost'] - df_bills['retrofit_cost']

    # Weight by EUSS sample weight
    for col in ['baseline_cost', 'retrofit_cost', 'bill_savings']:
        df_bills[f'weighted_{col}'] = df_bills[col] * df_bills['weight']

    # Aggregate by state
    grouped = df_bills.groupby('in.state').agg(
        home_count=('weight', 'size'),
        total_baseline_cost=('weighted_baseline_cost', 'sum'),
        total_retrofit_cost=('weighted_retrofit_cost', 'sum'),
        total_bill_savings=('weighted_bill_savings', 'sum'),
    ).reset_index().rename(columns={'in.state': 'state'})

    grouped['avg_savings_per_home'] = grouped['total_bill_savings'] / grouped['home_count']

    grouped['pct_bill_savings'] = np.where(
        grouped['total_baseline_cost'] > 0,
        grouped['total_bill_savings'] / grouped['total_baseline_cost'] * 100,
        np.nan
    )

    # Round for display
    for col in ['total_baseline_cost', 'total_retrofit_cost', 'total_bill_savings']:
        grouped[col] = grouped[col].round(0)
    grouped['avg_savings_per_home'] = grouped['avg_savings_per_home'].round(2)
    grouped['pct_bill_savings'] = grouped['pct_bill_savings'].round(2)

    if verbose:
        n_positive = (grouped['total_bill_savings'] > 0).sum()
        print(f"States where electrification saves money: {n_positive} / {len(grouped)}")
        print(f"National total bill savings: ${grouped['total_bill_savings'].sum():,.0f}")
        print(f"Mean % savings: {grouped['pct_bill_savings'].mean():.1f}%")

    return grouped.sort_values('pct_bill_savings', ascending=False).reset_index(drop=True)


# ============================================================================
# COMPUTE ACTUAL BILL SAVINGS (validation against ratio-based metric)
# ============================================================================

print(f"===== STEP 4b: ACTUAL BILL SAVINGS (MP{primary_mp}, gas homes, fuel-specific prices) =====")
df_bill_savings = compute_actual_bill_savings_by_state(
    df_baseline, df_upgrade_primary, df_prices_csv,
    fuel_filter='Natural Gas', verbose=True
)

print(f"\n--- Top 5 States (Highest % Bill Savings) ---")
print(df_bill_savings[['state', 'avg_savings_per_home', 'pct_bill_savings']].head(5).to_string(index=False))

print(f"\n--- Bottom 5 States ---")
print(df_bill_savings[['state', 'avg_savings_per_home', 'pct_bill_savings']].tail(5).to_string(index=False))

print("\n✓ STEP 4b COMPLETE")

---
# STEP 5: Change in Demand Under Adoption Scenario
---

**Goal:** Quantify demand change when homes adopt heat pumps. Computes **two distinct metrics:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Electricity demand change** | retrofit_elec_total − baseline_elec | Grid impact: how much MORE (or less) electricity the grid must supply |
| **Total site energy change** | retrofit_elec_total − baseline_all_fuels | Thermodynamic efficiency: total energy saved across all fuels |

**Retrofit electricity** includes both HP compressor and backup resistance heating.

**Why two metrics?**
- A gas-heated home switching to HP goes from ~0 kWh electric heating to ~7,000 kWh → **+7,000 kWh** electricity demand (grid stress signal)
- But it eliminates ~47,000 kWh of gas → **-40,000 kWh** total site energy (efficiency gain)
- Only reporting site energy change (-40K) hides the grid impact (+7K)

**Fuel filtering:** Optional `fuel_filter` parameter segments results by baseline fuel type.
Gas-to-electric homes show POSITIVE electricity demand change (grid needs more power).
Baseboard-to-HP homes show NEGATIVE electricity demand change (HP is more efficient than resistance).

**Standalone mode (100% adoption):** All applicable homes adopt. For tier-based analysis,
use the full TARE pipeline.

In [ ]:
def compute_scenario_demand(
    df_baseline: pd.DataFrame,
    df_upgrade: pd.DataFrame,
    fuel_filter: str = None,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Compute per-building heating demand change under 100% adoption scenario.

    Produces TWO change metrics for each home:
    - elec_demand_change_kwh: change in ELECTRICITY consumption only (grid impact)
    - site_energy_change_kwh: change in TOTAL site energy across all fuels (efficiency)

    Retrofit electricity includes HP compressor, backup resistance, and fan/pump.

    For gas-heated homes, electricity demand change is POSITIVE (grid stress signal):
    baseline electric heating ≈ 0, retrofit ≈ 5,000–15,000 kWh → large increase.
    For baseboard homes, electricity demand change is NEGATIVE (HP more efficient).

    Args:
        df_baseline: EUSS baseline DataFrame (indexed by bldg_id) with original
            EUSS columns including HEATING_FUEL_COLS, HP_BACKUP_ELEC_COL,
            HP_FANS_PUMPS_COL, 'in.state', 'in.heating_fuel', and 'weight'.
        df_upgrade: EUSS upgrade DataFrame (indexed by bldg_id, already filtered
            to applicability == True) with heating, backup, and fan/pump columns.
        fuel_filter: Filter to this heating fuel (e.g., 'Natural Gas').
            Set to None (default) to include all fuel types.
        verbose: Print diagnostic info.

    Returns:
        DataFrame (indexed by bldg_id) with columns: in.state, in.heating_fuel,
        weight, baseline_electric_kwh, baseline_heating_total_kwh,
        retrofit_electric_kwh, elec_demand_change_kwh, site_energy_change_kwh,
        and weighted_* versions of each change metric.
    """
    elec_col = 'out.electricity.heating.energy_consumption.kwh'
    bkup_col = HP_BACKUP_ELEC_COL
    fans_col = HP_FANS_PUMPS_COL

    # Baseline: electricity-only heating and total across all fuels
    baseline_total = df_baseline[HEATING_FUEL_COLS].sum(axis=1)

    # Retrofit: HP compressor + backup resistance + fan/pump
    retrofit_total_elec = (
        df_upgrade[elec_col].fillna(0)
        + df_upgrade[bkup_col].fillna(0)
        + df_upgrade[fans_col].fillna(0)
    )

    # Build per-building demand table (inner join on bldg_id)
    df_demand = pd.DataFrame({
        'in.state': df_baseline['in.state'],
        'in.heating_fuel': df_baseline['in.heating_fuel'],
        'weight': df_baseline['weight'],
        'baseline_electric_kwh': df_baseline[elec_col],
        'baseline_heating_total_kwh': baseline_total,
    }).join(
        retrofit_total_elec.rename('retrofit_electric_kwh'),
        how='inner'
    )

    # Optional fuel filter
    if fuel_filter is not None:
        n_before = len(df_demand)
        df_demand = df_demand[df_demand['in.heating_fuel'] == fuel_filter]
        if verbose:
            print(f"Filtered to '{fuel_filter}': {len(df_demand):,} / {n_before:,} homes")

    # ELECTRICITY demand change: what the GRID sees
    # Positive = grid must supply more electricity (gas-to-HP conversions)
    # Negative = grid supplies less electricity (baseboard-to-HP conversions)
    df_demand['elec_demand_change_kwh'] = (
        df_demand['retrofit_electric_kwh'] - df_demand['baseline_electric_kwh']
    )

    # TOTAL SITE ENERGY change: thermodynamic efficiency across all fuels
    # Negative = less total energy consumed (typical, since HP COP > 1)
    df_demand['site_energy_change_kwh'] = (
        df_demand['retrofit_electric_kwh'] - df_demand['baseline_heating_total_kwh']
    )

    # Weighted versions (using EUSS weight column)
    for col in ['baseline_electric_kwh', 'baseline_heating_total_kwh',
                'retrofit_electric_kwh', 'elec_demand_change_kwh',
                'site_energy_change_kwh']:
        df_demand[f'weighted_{col}'] = df_demand[col] * df_demand['weight']

    if verbose:
        fuel_label = fuel_filter if fuel_filter else 'all fuels'
        print(f"\n--- Demand Scenario Summary (100% adoption, {fuel_label}) ---")
        print(f"Total homes: {len(df_demand):,}")
        elec_change_gwh = df_demand['weighted_elec_demand_change_kwh'].sum() / 1e6
        site_change_gwh = df_demand['weighted_site_energy_change_kwh'].sum() / 1e6
        baseline_elec_gwh = df_demand['weighted_baseline_electric_kwh'].sum() / 1e6
        baseline_total_gwh = df_demand['weighted_baseline_heating_total_kwh'].sum() / 1e6
        print(f"Weighted baseline electric heating: {baseline_elec_gwh:,.1f} GWh")
        print(f"Weighted baseline total heating:    {baseline_total_gwh:,.1f} GWh")
        print(f"Weighted electricity demand change:  {elec_change_gwh:+,.1f} GWh (grid impact)")
        print(f"Weighted total site energy change:   {site_change_gwh:+,.1f} GWh (efficiency)")

    return df_demand


# ============================================================================
# COMPUTE DEMAND CHANGE FROM LOADED EUSS DATA
# ============================================================================

print(f"===== STEP 5a: COMPUTE SCENARIO DEMAND (MP{primary_mp}, 100% adoption, all fuels) =====")
df_demand = compute_scenario_demand(df_baseline, df_upgrade_primary, fuel_filter=None, verbose=True)

print(f"\n--- Sample rows (gas homes) ---")
gas_sample = df_demand[df_demand['in.heating_fuel'] == 'Natural Gas'].head(3)
print(gas_sample[['in.state', 'in.heating_fuel', 'baseline_electric_kwh',
                   'baseline_heating_total_kwh', 'retrofit_electric_kwh',
                   'elec_demand_change_kwh', 'site_energy_change_kwh']].to_string())

print(f"\n--- Sample rows (electric baseboard homes) ---")
elec_sample = df_demand[df_demand['in.heating_fuel'] == 'Electricity'].head(3)
print(elec_sample[['in.state', 'in.heating_fuel', 'baseline_electric_kwh',
                    'baseline_heating_total_kwh', 'retrofit_electric_kwh',
                    'elec_demand_change_kwh', 'site_energy_change_kwh']].to_string())

print("\n✓ STEP 5a COMPLETE")

In [ ]:
def aggregate_demand_by_state(
    df_demand: pd.DataFrame,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Aggregate per-building demand scenario results to state-level totals.

    Reports BOTH demand change metrics in GWh:
    - elec_change_gwh: grid impact (electricity-only change)
    - site_energy_change_gwh: total site energy change (all fuels)

    Args:
        df_demand: Per-building demand DataFrame from compute_scenario_demand().
            Required columns: in.state, weighted_baseline_electric_kwh,
            weighted_baseline_heating_total_kwh, weighted_retrofit_electric_kwh,
            weighted_elec_demand_change_kwh, weighted_site_energy_change_kwh.
        verbose: Print diagnostic info.

    Returns:
        DataFrame with columns: state, home_count, baseline_elec_gwh,
        baseline_total_gwh, retrofit_elec_gwh, elec_change_gwh,
        pct_elec_demand_change, site_energy_change_gwh, pct_site_energy_change.
    """
    grouped = df_demand.groupby('in.state').agg(
        home_count=('weight', 'size'),
        weighted_baseline_elec=('weighted_baseline_electric_kwh', 'sum'),
        weighted_baseline_total=('weighted_baseline_heating_total_kwh', 'sum'),
        weighted_retrofit_elec=('weighted_retrofit_electric_kwh', 'sum'),
        weighted_elec_change=('weighted_elec_demand_change_kwh', 'sum'),
        weighted_site_change=('weighted_site_energy_change_kwh', 'sum'),
    ).reset_index().rename(columns={'in.state': 'state'})

    # Convert kWh to GWh
    grouped['baseline_elec_gwh'] = grouped['weighted_baseline_elec'] / 1e6
    grouped['baseline_total_gwh'] = grouped['weighted_baseline_total'] / 1e6
    grouped['retrofit_elec_gwh'] = grouped['weighted_retrofit_elec'] / 1e6
    grouped['elec_change_gwh'] = grouped['weighted_elec_change'] / 1e6
    grouped['site_energy_change_gwh'] = grouped['weighted_site_change'] / 1e6

    # % change relative to appropriate baseline
    grouped['pct_elec_demand_change'] = np.where(
        grouped['weighted_baseline_elec'] != 0,
        grouped['weighted_elec_change'] / grouped['weighted_baseline_elec'] * 100,
        np.nan
    )
    grouped['pct_site_energy_change'] = np.where(
        grouped['weighted_baseline_total'] != 0,
        grouped['weighted_site_change'] / grouped['weighted_baseline_total'] * 100,
        np.nan
    )

    # --- Validation: demand accounting ---
    total_elec_sum = df_demand['weighted_elec_demand_change_kwh'].sum()
    total_elec_agg = grouped['weighted_elec_change'].sum()
    accounting_ok = np.isclose(total_elec_sum, total_elec_agg, rtol=1e-6)
    if not accounting_ok:
        print(f"⚠ DEMAND ACCOUNTING MISMATCH: sum={total_elec_sum:.0f}, agg={total_elec_agg:.0f}")
    elif verbose:
        print(f"✓ Demand accounting check passed")

    # Select output columns
    result = grouped[[
        'state', 'home_count',
        'baseline_elec_gwh', 'baseline_total_gwh', 'retrofit_elec_gwh',
        'elec_change_gwh', 'pct_elec_demand_change',
        'site_energy_change_gwh', 'pct_site_energy_change'
    ]].copy()

    # Round for display
    for col in ['baseline_elec_gwh', 'baseline_total_gwh', 'retrofit_elec_gwh',
                'elec_change_gwh', 'site_energy_change_gwh']:
        result[col] = result[col].round(2)
    result['pct_elec_demand_change'] = result['pct_elec_demand_change'].round(2)
    result['pct_site_energy_change'] = result['pct_site_energy_change'].round(2)

    if verbose:
        print(f"\n--- State-Level Demand Summary ---")
        print(f"States: {len(result)}")
        print(f"Total baseline electric:     {result['baseline_elec_gwh'].sum():.1f} GWh")
        print(f"Total baseline (all fuels):  {result['baseline_total_gwh'].sum():.1f} GWh")
        print(f"Total retrofit electric:     {result['retrofit_elec_gwh'].sum():.1f} GWh")
        print(f"Total elec demand change:    {result['elec_change_gwh'].sum():+.1f} GWh (grid impact)")
        print(f"Total site energy change:    {result['site_energy_change_gwh'].sum():+.1f} GWh (efficiency)")

    return result.sort_values('elec_change_gwh', ascending=False).reset_index(drop=True)


# ============================================================================
# AGGREGATE DEMAND BY STATE
# ============================================================================

print("===== STEP 5b: AGGREGATE DEMAND BY STATE =====")
df_demand_state = aggregate_demand_by_state(df_demand, verbose=True)

print(f"\n--- Top 5 States (Largest ELECTRICITY Demand Increase) ---")
print(df_demand_state[['state', 'elec_change_gwh', 'pct_elec_demand_change',
                        'site_energy_change_gwh', 'pct_site_energy_change']].head(5).to_string(index=False))

print(f"\n--- Bottom 5 States (Smallest Electricity Demand Change) ---")
print(df_demand_state[['state', 'elec_change_gwh', 'pct_elec_demand_change',
                        'site_energy_change_gwh', 'pct_site_energy_change']].tail(5).to_string(index=False))

print("\n✓ STEP 5b COMPLETE")

---
# STEP 6: Geospatial Visualization
---

**Goal:** Generate choropleth maps of spark gap, bill impact ratio, and thermal COP by state.

| Sub-step | Description |
|----------|------------|
| **6a** | Load Census TIGER state boundaries, merge with analysis results |
| **6b** | Generalized `create_choropleth_map()` — spark gap, bill impact ratio, COP |
| **6c** | Optional demand change map (if `df_demand_state` is available) |

In [ ]:
# ============================================================================
# STEP 6a: Load State Boundaries and Merge with Analysis Data
# ============================================================================

def prepare_state_geodataframe(
    gdf_states: gpd.GeoDataFrame,
    df_analysis: pd.DataFrame,
    merge_col: str = 'state',
    exclude_territories: Optional[list] = None
) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Merge analysis results with state geometries and prepare for choropleth mapping.

    Handles different Census shapefile column naming conventions
    (STUSPS, STUSAB, STATE_ABBR) by auto-detecting the state abbreviation column.
    Reprojects to US Albers Equal Area (ESRI:102003) and splits into
    CONUS and Alaska GeoDataFrames for inset plotting.

    Args:
        gdf_states: GeoDataFrame of US state boundaries (any CRS).
        df_analysis: DataFrame with a state abbreviation column and analysis columns.
        merge_col: Column name in df_analysis containing 2-letter state abbreviations.
        exclude_territories: State/territory codes to exclude (default: PR, VI, GU, AS, MP).

    Returns:
        Tuple of (gdf_all, gdf_conus, gdf_alaska) — all in ESRI:102003.
    """
    if exclude_territories is None:
        exclude_territories = ['PR', 'VI', 'GU', 'AS', 'MP']

    # Auto-detect state abbreviation column
    state_col = None
    for col in ['STUSPS', 'STUSAB', 'STATE_ABBR']:
        if col in gdf_states.columns:
            state_col = col
            break

    if state_col is None:
        raise ValueError(
            f"No state abbreviation column found. Available columns: {list(gdf_states.columns)}"
        )

    # Reproject to US Albers Equal Area for accurate area representation
    gdf_states = gdf_states.to_crs('ESRI:102003')

    # Merge analysis data with geometries
    gdf = gdf_states.merge(
        df_analysis,
        left_on=state_col,
        right_on=merge_col,
        how='left'
    )

    # Filter: exclude territories and rows with no analysis data
    # Use the first non-id numeric column to detect missing data
    numeric_cols = df_analysis.select_dtypes(include='number').columns.tolist()
    filter_col = numeric_cols[0] if numeric_cols else merge_col

    gdf_filtered = gdf[
        (~gdf[state_col].isin(exclude_territories)) &
        (gdf[filter_col].notna())
    ].copy()

    gdf_alaska = gdf_filtered[gdf_filtered[state_col] == 'AK'].copy()
    gdf_conus = gdf_filtered[
        ~gdf_filtered[state_col].isin(['AK', 'HI'])
    ].copy()

    return gdf_filtered, gdf_conus, gdf_alaska


# ─── Load shapefile ─────────────────────────────────────────────────────────
SHAPEFILE_PATH = os.path.join(
    PROJECT_ROOT, "cmu_tare_model", "data", "shapefiles", "US_state_2015.shp"
)

gdf_conus = None
gdf_alaska = None

try:
    print(f"Loading state shapefile from: {SHAPEFILE_PATH}")
    gdf_states_raw = gpd.read_file(SHAPEFILE_PATH)
    print(f"  Loaded {len(gdf_states_raw)} state/territory polygons")
    print(f"  Columns: {list(gdf_states_raw.columns)}")

    _, gdf_conus, gdf_alaska = prepare_state_geodataframe(
        gdf_states_raw,
        df_spark,
        merge_col='state'
    )
    print(f"  CONUS states: {len(gdf_conus)}")
    print(f"  Alaska polygons: {len(gdf_alaska)}")
    print("✓ STEP 6a COMPLETE - State geodataframe prepared")

except FileNotFoundError:
    print(f"⚠ WARNING: Shapefile not found at {SHAPEFILE_PATH}")
    print("  Skipping geospatial visualization. To enable, add the shapefile to:")
    print(f"  {os.path.dirname(SHAPEFILE_PATH)}")
except Exception as e:
    print(f"⚠ WARNING: Could not load shapefile: {e}")
    print("  Skipping geospatial visualization.")


---
## STEP 6b: Create Choropleth Maps
---

The `create_choropleth_map()` function is a generalized version of the `create_price_ratio_map()`
from the standalone `visualize_state_price_ratios.ipynb` notebook. It accepts any numeric column
from the merged GeoDataFrame as the mapping target.

In [ ]:
# ============================================================================
# STEP 6b: Generalized Choropleth Map Function + Generate Maps
# ============================================================================

def create_choropleth_map(
    gdf_conus: gpd.GeoDataFrame,
    gdf_alaska: gpd.GeoDataFrame,
    column: str,
    title: str,
    cbar_label: str,
    year: int = 2022,
    output_path: Optional[str] = None,
    figsize: Tuple[int, int] = (16, 10),
    dpi: int = 300,
    cmap: str = 'RdYlBu_r',
    show_plot: bool = True
) -> Optional[str]:
    """
    Create a choropleth map of any numeric column from state-level GeoDataFrames.

    Renders a continental US main panel with an Alaska inset in the lower-left,
    a vertical colorbar on the right, an interpretation annotation, and a data
    source citation footer.

    Args:
        gdf_conus: GeoDataFrame of continental US states (ESRI:102003).
        gdf_alaska: GeoDataFrame of Alaska polygon(s) (ESRI:102003).
        column: Name of the numeric column to map (e.g., 'spark_gap').
        title: Map title string.
        cbar_label: Label for the colorbar axis.
        year: Data year shown in the title/footer (default 2022).
        output_path: If provided, save the figure to this path at `dpi` DPI.
        figsize: Figure size in inches (width, height).
        dpi: Resolution for saved figure.
        cmap: Matplotlib colormap name.
        show_plot: If True, display inline; if False, close figure after saving.

    Returns:
        output_path if saved, otherwise None.
    """
    all_values = pd.concat([gdf_conus[column], gdf_alaska[column]]).dropna()
    vmin, vmax = all_values.min(), all_values.max()

    fig = plt.figure(figsize=figsize, facecolor='white')
    ax_main = fig.add_axes([0.05, 0.1, 0.75, 0.8])
    ax_alaska = fig.add_axes([0.05, 0.1, 0.22, 0.25])

    # Continental US
    gdf_conus.plot(
        column=column, ax=ax_main, cmap=cmap,
        edgecolor='black', linewidth=0.5,
        vmin=vmin, vmax=vmax, legend=False
    )
    ax_main.set_axis_off()
    ax_main.set_title(title, fontsize=18, fontweight='bold', pad=20)

    # Alaska inset
    gdf_alaska.plot(
        column=column, ax=ax_alaska, cmap=cmap,
        edgecolor='black', linewidth=0.5,
        vmin=vmin, vmax=vmax, legend=False
    )
    ax_alaska.set_axis_off()
    ax_alaska.set_title('Alaska', fontsize=12, fontweight='bold')

    # Colorbar
    cax = fig.add_axes([0.82, 0.15, 0.03, 0.6])
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(cbar_label, fontsize=12, labelpad=15)

    # Footer citation
    fig.text(
        0.5, 0.02,
        f'Data Source: NREL EUSS + EIA Average Residential Prices ({year})',
        fontsize=9, ha='center', color='gray'
    )

    if output_path:
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"  Saved: {output_path}")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return output_path if output_path else None


# ─── Generate maps (only if geodataframe was loaded successfully) ────────────
if gdf_conus is not None and gdf_alaska is not None:
    print("Generating choropleth maps...")

    # Map 1: Spark Gap
    spark_gap_path = os.path.join(PROJECT_ROOT, "state_spark_gap_map_2022.png")
    create_choropleth_map(
        gdf_conus, gdf_alaska,
        column='spark_gap',
        title='Electricity-to-Natural Gas Spark Gap by State (2022)',
        cbar_label='Spark Gap\n(electricity price ÷ gas price, $/MMBTU basis)',
        year=2022,
        output_path=spark_gap_path,
        cmap='RdYlBu_r',
        show_plot=True
    )

    # Map 2: Bill Impact Ratio
    bill_impact_path = os.path.join(PROJECT_ROOT, "state_bill_impact_ratio_map_2022.png")
    create_choropleth_map(
        gdf_conus, gdf_alaska,
        column='bill_impact_ratio',
        title='Heat Pump Bill Impact Ratio by State (2022)\n(ratio < 1 = HP saves money; ratio > 1 = HP costs more)',
        cbar_label='Bill Impact Ratio\n(HP operating cost / gas furnace cost)',
        year=2022,
        output_path=bill_impact_path,
        cmap='RdYlGn_r',
        show_plot=True
    )

    # Map 3: Thermal COP
    thermal_cop_path = os.path.join(PROJECT_ROOT, "state_thermal_cop_map_2022.png")
    create_choropleth_map(
        gdf_conus, gdf_alaska,
        column='thermal_cop',
        title='Heat Pump Thermal COP by State (2022)',
        cbar_label='Thermal COP\n(heat delivered / electricity consumed)',
        year=2022,
        output_path=thermal_cop_path,
        cmap='YlOrRd',
        show_plot=True
    )

    print("✓ STEP 6b COMPLETE - Choropleth maps generated")
else:
    print("⚠ STEP 6b SKIPPED - No geodataframe available (shapefile not loaded in Step 6a)")


---
## STEP 6c: Demand Change Map (Optional)
---

Maps `elec_change_gwh` from `df_demand_state` using a diverging colormap centered at zero.
Only runs if `df_demand_state` is available and the geodataframe was loaded in Step 6a.

In [ ]:
# ============================================================================
# STEP 6c: Demand Change Choropleth Map (Optional)
# ============================================================================

if gdf_conus is not None and gdf_alaska is not None:
    try:
        # Check that df_demand_state is defined (NameError if not)
        try:
            _ = df_demand_state
        except NameError:
            print("⚠ STEP 6c SKIPPED - df_demand_state not available")
            raise

        if df_demand_state is not None:
            # Merge demand data with geometries
            _, gdf_demand_conus, gdf_demand_alaska = prepare_state_geodataframe(
                gdf_states_raw,
                df_demand_state,
                merge_col='state'
            )

            demand_path = os.path.join(PROJECT_ROOT, "state_elec_demand_change_map_2022.png")
            create_choropleth_map(
                gdf_demand_conus, gdf_demand_alaska,
                column='elec_change_gwh',
                title='Electricity Demand Change Under 100% HP Adoption by State (2022)',
                cbar_label='Electricity Demand Change (GWh)\n(positive = more grid electricity needed)',
                year=2022,
                output_path=demand_path,
                cmap='coolwarm',
                show_plot=True
            )
            print("✓ STEP 6c COMPLETE - Demand change map generated")
        else:
            print("⚠ STEP 6c SKIPPED - df_demand_state is None")
    except NameError:
        pass
    except Exception as e:
        print(f"⚠ STEP 6c ERROR: {e}")
else:
    print("⚠ STEP 6c SKIPPED - No geodataframe available (shapefile not loaded in Step 6a)")


---
# Display Results
---

View the complete results tables: price ratios, thermal COP, baseline AFUE, bill impact ratio, and demand change by state.

In [ ]:
print("===== STATE PRICE RATIO TABLE — CSV Nominal Prices (2022) =====\n")
display(df_prices_csv)

for mp in selected_mps:
    print(f"\n===== THERMAL COP & BASELINE AFUE BY STATE (MP{mp}, Natural Gas homes only) =====\n")
    display(cop_results[mp].sort_values('thermal_cop', ascending=False)[
        ['state', 'thermal_cop', 'baseline_afue', 'fans_pumps_pct', 'home_count']
    ])

print(f"\n===== COMPOSITE SPARK GAP METRICS (MP{primary_mp}, gas homes) =====\n")
display(df_spark[['state', 'state_name', 'spark_gap', 'thermal_cop', 'baseline_afue', 'bill_impact_ratio']])

print(f"\n===== ACTUAL BILL SAVINGS (MP{primary_mp}, gas homes, fuel-specific prices) =====\n")
display(df_bill_savings[['state', 'avg_savings_per_home', 'pct_bill_savings']])

print(f"\n===== STATE-LEVEL DEMAND CHANGE (MP{primary_mp}, GWh, all fuels, 100% adoption) =====\n")
display(df_demand_state[['state', 'home_count', 'elec_change_gwh',
                          'pct_elec_demand_change', 'site_energy_change_gwh',
                          'pct_site_energy_change']])

---
# Summary
---

## Functions Defined in This Notebook

| Step | Function | Purpose |
|------|----------|---------|
| 1b | `load_euss_baseline()` | Load EUSS baseline CSV → filter to occupied SF homes |
| 1b | `load_euss_upgrade()` | Load EUSS upgrade CSV → filter to applicable occupied SF homes |
| 2 | `calculate_price_ratios()` | Load EIA nominal prices from CSV → spark gap by state |
| 2b | `get_state_fuel_prices_from_lookup()` | Extract USD2023 prices from TARE lookup dict → spark gap by state |
| 3 | `compute_thermal_cop_by_state()` | Thermal COP from heating load + baseline AFUE from data, with `fuel_filter` |
| 4 | `compute_spark_gap_metrics()` | Merge prices + COP/AFUE → `bill_impact_ratio` (HP cost / furnace cost) |
| 4b | `compute_actual_bill_savings_by_state()` | Per-building bill comparison with actual fuel-specific prices |
| 5a | `compute_scenario_demand()` | Per-building demand: **elec_demand_change** + **site_energy_change** |
| 5b | `aggregate_demand_by_state()` | State-level demand (GWh): grid impact + efficiency metrics |
| 6a | `prepare_state_geodataframe()` | Merge analysis results with Census TIGER state boundaries |
| 6b | `create_choropleth_map()` | Generalized choropleth mapping for any metric (spark gap, bill impact ratio, COP, demand) |

## Key Design Decisions

1. **Thermal COP from heating load:** Uses `out.load.heating.energy_delivered.kbtu` to compute
   true heat pump COP (Q_delivered / E_hp_total) following the literature method, rather than
   the opaque "system COP" ratio that conflated HP and furnace efficiencies. Expected range: 2.0–4.5.

2. **Fan/pump energy in COP denominator:** The denominator includes all three electricity
   components: HP compressor, backup resistance, AND supply fan/pump electricity
   (`out.electricity.heating_fans_pumps.energy_consumption.kwh`). Fan energy is 5–30% of
   total in warm states, and omitting it was the root cause of inflated COP values (6.0+).

3. **Data-derived baseline AFUE:** Computes furnace efficiency directly from simulation data
   (Q_delivered / F_gas) rather than using an assumed constant. The AFUE appears exactly once
   in the bill impact ratio formula, eliminating the prior double-counting bug.

4. **Multi-MP batch support:** Supports multiple measure packages (MP3, MP4) via
   `VALID_MENU_MPS` from constants. When called from `tare_scenarios_v2_2.ipynb`,
   `input_measure_package` selects a single MP; standalone mode processes all `SELECTABLE_MPS`.

5. **Fuel filtering:** COP and spark gap metrics default to `fuel_filter='Natural Gas'`
   because the spark gap (elec_price / gas_price) is specifically about gas-to-electric economics.

6. **Dual demand metrics:** `elec_demand_change` shows grid impact (positive for gas→HP),
   `site_energy_change` shows thermodynamic efficiency (always negative since HP COP > 1).

7. **Actual bill savings validation:** `compute_actual_bill_savings_by_state()` prices each fuel
   at its correct rate per building, validating the ratio-based `bill_impact_ratio`.

## Standalone Usage

```python
df_baseline = load_euss_baseline()
upgrade_data = {mp: load_euss_upgrade(mp_to_upgrade(mp)) for mp in selected_mps}

df_prices = calculate_price_ratios(FUEL_PRICES_PATH, year=2022)
df_cop = compute_thermal_cop_by_state(df_baseline, upgrade_data[4], fuel_filter='Natural Gas', verbose=True)
df_spark = compute_spark_gap_metrics(df_prices, df_cop, verbose=True)
df_bill_savings = compute_actual_bill_savings_by_state(df_baseline, upgrade_data[4], df_prices, verbose=True)

df_demand = compute_scenario_demand(df_baseline, upgrade_data[4], verbose=True)
df_demand_state = aggregate_demand_by_state(df_demand, verbose=True)
```

## Output Files

- `state_spark_gap_map_{year}.png`
- `state_bill_impact_ratio_map_{year}.png`
- `state_thermal_cop_map_{year}.png`
- `state_elec_demand_change_map_{year}.png` (if demand data available)

## Deferred / TODO
- **10-Year Average Prices:** Load updated 10-year average residential fuel price data (awaiting data)
- **AFUE Sensitivity:** `compute_spark_gap_sensitivity_by_afue()` — override the data-derived AFUE
  with hypothetical values (0.80, 0.92, 0.96) to test how incumbent furnace efficiency
  affects adoption economics. The `DEFAULT_FURNACE_AFUE` constant (currently commented out)
  is reserved for this analysis.
- **Propane/Fuel Oil Prices:** Add price data for propane and fuel oil to `compute_actual_bill_savings_by_state()`.
- **Tier-Based Adoption:** Integrate with `determine_adoption_potential_sensitivity.py` for
  tier-weighted demand scenarios (Tier 1/2 adopters vs. Tier 3/4 non-adopters).
- **Validation Suite:** Validate thermal COP range (2.0–4.5), COP-HDD correlation, PA case study.
- **Peak Load / Timeseries:** Hourly demand profiles from OEDI EUSS timeseries data.